# 11 Training Data Optimization

연구 목적별 통합 Notebook — 승인된 매핑, 원본 스크립트 및 결과는 변경하지 않는다. 1차 Fine-Tuning까지 포함. 2차 Fine-Tuning 제외.

## 연구 구성
- [`exp_cmb010_targeted_handmade_train.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/scripts/exp_cmb010_targeted_handmade_train.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/logs)
- [`exp_cmb011_classwise_training.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/scripts/exp_cmb011_classwise_training.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/logs)
- [`exp_cmb010_aggregate.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/scripts/exp_cmb010_aggregate.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/logs)
- [`exp_cmb011_aggregate.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/scripts/exp_cmb011_aggregate.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/logs)

## 사용법
원본 코드 셀은 `%%archive_source`로 등록하는 실제 원본 코드이며, 함수명 충돌을 막기 위해 현재 커널에서 임의 실행하지 않는다. 맨 아래의 실행 셀에서 원본 스크립트/Notebook을 고정 커밋의 **별도 checkout**에서 실행한다. 각 실험의 실행 인자·입력·사전 조건은 위 원본 로그/코드로 확인한다. 기존 로그·결과와 새 출력이 섞이지 않도록 독립 복사본을 사용한다. Original Test/교수님 Test를 포함한 파일은 실행 환경에 두지 않는다. 과거 스크립트가 해당 자료를 요구할 경우 실행하지 않는다. 자동 재실행 및 성능 동등성은 미검증.

원본 실행을 위한 로그 디렉터리를 별도로 지정한다.

In [ ]:
# 공통 실행 제어 — Run All은 원본 코드 등록만 하며 학습하지 않는다.
from pathlib import Path
import hashlib, json, subprocess, sys, datetime
from IPython import get_ipython
SOURCE_CELLS = {}
def archive_source(line, cell):
    SOURCE_CELLS[line.strip()] = cell
get_ipython().register_magic_function(archive_source, magic_kind="cell", magic_name="archive_source")
# 실행 시에만 설정: 기존 작업 트리가 아닌 임시 checkout (git worktree add --detach ... <ref>)
WORKTREE = None
LOG_DIR = None
RUN_PLAN = []  # 예: [("원본/스크립트.py", ["--인자", "값"])]
ALLOW_EXECUTION = False

def blob_sha(data):
    return hashlib.sha1(f"blob {len(data)}\0".encode()+data).hexdigest()

def get_verified_source(path):
    if WORKTREE is None:
        raise ValueError("해당 커밋의 별도 checkout 경로 WORKTREE를 지정하세요.")
    root=Path(WORKTREE).expanduser().resolve()
    entry=SOURCE_INDEX[path]
    head=subprocess.check_output(["git","-C",str(root),"rev-parse","HEAD"],text=True).strip()
    if head != entry["ref"]:
        raise ValueError(f"checkout ref 불일치: {head} != {entry['ref']}")
    p=root/path
    if blob_sha(p.read_bytes()) != entry["sha"]:
        raise ValueError(f"원본 Git blob 불일치: {path}")
    if path in SOURCE_CELLS:
        if p.read_text(encoding="utf-8-sig").rstrip("\n") != SOURCE_CELLS[path].rstrip("\n").lstrip("\ufeff"):
            raise ValueError(f"Notebook 보존 코드와 원본이 다릅니다: {path}")
    return root,p

def show_original(path):
    from IPython.display import display, Code
    _,p=get_verified_source(path)
    display(Code(p.read_text(encoding="utf-8"),language="python" if p.suffix==".py" else "json"))

def execute_original(path, args):
    if not ALLOW_EXECUTION:
        raise RuntimeError("명시적 실행 승인이 필요합니다.")
    root,p=get_verified_source(path)
    if LOG_DIR is None:
        raise ValueError("새 외부 LOG_DIR를 지정하세요.")
    logs=Path(LOG_DIR).expanduser().resolve();logs.mkdir(parents=True,exist_ok=True)
    if logs==root or root in logs.parents:
        raise ValueError("LOG_DIR는 checkout 밖에 두세요.")
    stamp=datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    if p.suffix==".ipynb":
        cmd=[sys.executable,"-m","jupyter","nbconvert","--to","notebook","--execute",str(p),"--output-dir",str(logs),"--output",f"{p.stem}_{stamp}.ipynb",*args]
    else:
        cmd=[sys.executable,str(p),*args]
    log=logs/f"{p.stem}_{stamp}.log"
    with log.open("x",encoding="utf-8") as f:
        f.write(json.dumps({"source":path,**SOURCE_INDEX[path],"command":cmd,"cwd":str(p.parent)},ensure_ascii=False)+"\n");f.flush()
        result=subprocess.run(cmd,cwd=p.parent,stdout=f,stderr=subprocess.STDOUT)
        f.write(f"\nreturncode={result.returncode}\n")
    print(log)
    result.check_returncode()


In [ ]:
SOURCE_INDEX = {'05_model_optimize/MLP/Combined/scripts/exp_cmb010_targeted_handmade_train.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': '2c6ec2d2bc41dd1c47a426e32fee03ad4cd52c55'}, '05_model_optimize/MLP/Combined/scripts/exp_cmb011_classwise_training.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': 'da410a8c9cb25df8d977bb821c4a9ed263ddae79'}, '05_model_optimize/MLP/Combined/scripts/exp_cmb010_aggregate.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': 'f0d3a22452059adbcd63c6ec01eb801ddf88f63b'}, '05_model_optimize/MLP/Combined/scripts/exp_cmb011_aggregate.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': 'e897d9821753d6579701efb588ce7560571f998d'}}

## exp_cmb010_targeted_handmade_train

원본: [05_model_optimize/MLP/Combined/scripts/exp_cmb010_targeted_handmade_train.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/scripts/exp_cmb010_targeted_handmade_train.py)  
Git blob: `2c6ec2d2bc41dd1c47a426e32fee03ad4cd52c55`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/logs)

EXP-CMB-010: Compare baseline Combined Batch128 vs adding new 480 targeted handwritings.

No external438, Original Test, Professor Test or held-out Writer is used to fit.
Treat all 480 as ONE unknown-writer cohort, used only in fit for the add480 arm.
Model/preprocessing/official augment are otherwise the CMB008/CMB009 exact reference.

CLI 인자 (정적 추출): `--arm`, `--seed`, `--heldout`, `--out`

In [ ]:
%%archive_source 05_model_optimize/MLP/Combined/scripts/exp_cmb010_targeted_handmade_train.py
#!/usr/bin/env python3
"""EXP-CMB-010: Compare baseline Combined Batch128 vs adding new 480 targeted handwritings.

No external438, Original Test, Professor Test or held-out Writer is used to fit.
Treat all 480 as ONE unknown-writer cohort, used only in fit for the add480 arm.
Model/preprocessing/official augment are otherwise the CMB008/CMB009 exact reference.
"""
from __future__ import annotations
import argparse,csv,hashlib,json,time,warnings
from pathlib import Path
import joblib,numpy as np,sklearn
from sklearn.pipeline import Pipeline
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import f1_score,confusion_matrix
from exp_cmb008_s5_early_stopping import (
    ROOT,WRITERS,LABELS,V3_SHA,HANDMADE_SHA,
    load_official,load_writers,PixelScaler255,prepareraw,process
)
EXTRA=ROOT/"02_data"/"Hand_made_extra"/"CMB010_7_8_9_B_480.npz"
EXPECTED_SHA="3d1686c67d7bb5c6193384eca1aa9218213c842252baa5b7e172698743b83fc7"
TARGET_COUNTS={7:160,8:80,9:160,11:80}
OFFICIAL=ROOT/"02_data"/"datasets"/"MNIST_EMNIST_official_split_112000.npz"
HANDMADE=ROOT/"02_data"/"Hand_made"
PAIRS=((7,9,"7_to_9"),(9,7,"9_to_7"),(11,8,"B_to_8"),
       (8,11,"8_to_B"),(0,13,"0_to_D"),(13,0,"D_to_0"),
       (7,10,"7_to_A"),(4,15,"4_to_F"))

def sha_file(path):
    h=hashlib.sha256()
    with path.open("rb") as f:
        for b in iter(lambda:f.read(1024*1024),b""):h.update(b)
    return h.hexdigest()

def validate_extra():
    assert sha_file(EXTRA)==EXPECTED_SHA,"New input NPZ unexpectedly changed"
    with np.load(EXTRA,allow_pickle=False) as z:
        required={"X","y","images","labels","label_text","class_names","available_classes","source_image","cell_row_col"}
        assert required.issubset(z.files),(required-set(z.files))
        x=np.asarray(z["X"]);y=np.asarray(z["y"],dtype=np.int32)
        assert x.shape==(480,784) and x.dtype==np.uint8 and y.shape==(480,)
        assert np.array_equal(x,z["images"].reshape(480,784))
        assert np.array_equal(y,z["labels"])
        assert np.array_equal(z["label_text"],np.asarray([LABELS[k] for k in y]))
        assert np.array_equal(z["class_names"],np.asarray(list(LABELS)))
        assert {int(c):int(np.sum(y==c)) for c in sorted(set(y))}==TARGET_COUNTS
        assert len({row.tobytes() for row in x})==480,"Duplicate raw image"
        img=x.reshape(-1,28,28)
        mask=img>=32
        assert np.all(mask.reshape(480,-1).sum(axis=1)>=12),"Missing foreground"
        assert not(np.any(mask[:,0,:]) or np.any(mask[:,-1,:]) or
                   np.any(mask[:,:,0]) or np.any(mask[:,:,-1])),"Foreground clipped at edge"
        assert int(img.max())==206
        assert "writer_id" not in z.files and "writer_ids" not in z.files
    return x,y

def dump_csv(path,records):
    with path.open("w",newline="",encoding="utf-8-sig") as f:
        w=csv.DictWriter(f,fieldnames=list(records[0]))
        w.writeheader();w.writerows(records)

def metrics(y,p):
    cm=confusion_matrix(y,p,labels=np.arange(16))
    assert cm.sum()==len(y)
    return {"n":int(len(y)),"correct":int(np.sum(y==p)),
        "accuracy":float(np.mean(y==p)),
        "macro_f1":float(f1_score(y,p,average="macro",labels=np.arange(16),zero_division=0)),
        "class_support":{LABELS[c]:int(cm[c].sum()) for c in range(16)},
        "class_errors":{LABELS[c]:int(cm[c].sum()-cm[c,c]) for c in range(16)},
        "pairs":{name:int(cm[a,b]) for a,b,name in PAIRS},
        "digit_correct":int(np.sum(p[y<10]==y[y<10])),
        "letter_correct":int(np.sum(p[y>=10]==y[y>=10]))}

def main():
    ap=argparse.ArgumentParser()
    ap.add_argument("--arm",choices=("base","add480"),required=True)
    ap.add_argument("--seed",choices=(42,43,44),type=int,required=True)
    ap.add_argument("--heldout",choices=WRITERS,type=int,required=True)
    ap.add_argument("--out",type=Path,required=True)
    a=ap.parse_args()
    if a.out.exists():raise FileExistsError(a.out)
    a.out.mkdir(parents=True)
    begin=time.perf_counter()
    off=load_official(OFFICIAL);writers=load_writers(HANDMADE)
    assert off["sha256"]==V3_SHA
    for w in WRITERS:assert writers[w]["sha256"]==HANDMADE_SHA[w]
    new_raw,new_y=validate_extra()
    main_x,wr_x,base_aug,_=prepareraw(off,writers,"scale30_stroke28")
    remain=[w for w in WRITERS if w!=a.heldout]
    fit_x=[main_x]+[wr_x[w] for w in remain]
    fit_y=[np.asarray(off["y_train"],dtype=np.int32)]+[np.asarray(writers[w]["y"],dtype=np.int32) for w in remain]
    if a.arm=="add480":
        # New cohort is never used for writer-held-out evaluation or OfficialVal.
        # Do not change or augment labels; all are unambiguously encoded in NPZ.
        new_x=process(new_raw,"com",20)
        fit_x.append(new_x)
        fit_y.append(new_y)
    fit_x=np.concatenate(fit_x)
    fit_y=np.concatenate(fit_y)
    expected=78720+(480 if a.arm=="add480" else 0)
    assert fit_x.shape==(expected,784) and fit_y.shape==(expected,)
    held_x=wr_x[a.heldout]
    held_y=np.asarray(writers[a.heldout]["y"],dtype=int)
    val_x=process(off["X_val"],"com",20)
    val_y=np.asarray(off["y_val"],dtype=int)
    assert len(held_y)==640 and len(val_y)==19200
    mlp=MLPClassifier(random_state=a.seed,hidden_layer_sizes=(256,),
        alpha=.0001,learning_rate_init=.001,solver="adam",activation="relu",
        early_stopping=False,batch_size=128,max_iter=200,
        n_iter_no_change=10,tol=.0001)
    model=Pipeline([("scale_0_1",PixelScaler255()),("model",mlp)])
    with warnings.catch_warnings(record=True) as got:
        warnings.simplefilter("always")
        tick=time.perf_counter()
        model.fit(fit_x,fit_y)
        fit_sec=time.perf_counter()-tick
    assert np.array_equal(mlp.classes_,np.arange(16))
    op=np.asarray(model.predict(held_x),dtype=int)
    vp=np.asarray(model.predict(val_x),dtype=int)
    oof=metrics(held_y,op)
    val=metrics(val_y,vp)
    dump_csv(a.out/"heldout_predictions.csv",
        [{"arm":a.arm,"seed":a.seed,"heldout":a.heldout,
          "index":i,"actual":LABELS[int(held_y[i])],
          "pred":LABELS[int(op[i])]} for i in range(640)])
    # Optional fitted models preserved only in GitHub Actions artifacts, not deployed.
    joblib.dump(model,a.out/"candidate_model.joblib",compress=3)
    m={"experiment":"EXP-CMB-010","arm":a.arm,"seed":a.seed,"heldout":a.heldout,
       "official_sha256":off["sha256"],
       "old_writers_sha256":{str(w):writers[w]["sha256"] for w in WRITERS},
       "added_source_sha256":EXPECTED_SHA,
       "new_cohort_writer_id_known":False,"new_cohort_grouped_as_one":True,
       "added_images_count":480 if a.arm=="add480" else 0,
       "added_by_class":{LABELS[k]:v if a.arm=="add480" else 0 for k,v in TARGET_COUNTS.items()},
       "training_samples":expected,"heldout_samples":len(held_y),
       "official_val_samples":len(val_y),"heldout_in_fit":False,
       "new_cohort_in_fit":a.arm=="add480",
       "new_cohort_tested_as_independent":False,
       "external_438_used":False,"original_test_used":False,"professor_test_used":False,
       "frozen":{"hidden_layer_sizes":[256],"alpha":.0001,"learning_rate_init":.001,
                 "activation":"relu","solver":"adam","early_stopping":False,
                 "batch_size":128,"max_iter":200,"n_iter_no_change":10,"tol":.0001,
                 "official_aug":"scale30_stroke28","inference_preprocess":"CoM"},
       "augmentation":base_aug,"oof":oof,"official_val":val,
       "n_iter":int(mlp.n_iter_),"fit_seconds":float(fit_sec),
       "total_seconds":float(time.perf_counter()-begin),
       "sklearn_version":sklearn.__version__,
       "warnings":[str(w.message) for w in got],
       "model_sha256":sha_file(a.out/"candidate_model.joblib")}
    (a.out/"manifest.json").write_text(json.dumps(m,ensure_ascii=False,indent=2)+"\n",encoding="utf-8")
    print("CMB010_FINISHED",json.dumps({"arm":a.arm,"seed":a.seed,
        "heldout":a.heldout,"fit_count":expected,"oof":oof["accuracy"],
        "val":val["accuracy"],"7_to_9":oof["pairs"]["7_to_9"],
        "B_to_8":oof["pairs"]["B_to_8"],"fit_seconds":round(fit_sec,1)}),flush=True)

if __name__=="__main__":main()


## exp_cmb011_classwise_training

원본: [05_model_optimize/MLP/Combined/scripts/exp_cmb011_classwise_training.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/scripts/exp_cmb011_classwise_training.py)  
Git blob: `da410a8c9cb25df8d977bb821c4a9ed263ddae79`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/logs)

EXP-CMB-011: matched class-subset real handwriting vs official-train duplication controls.

Seven arms: base, add79 (320), addB8 (160), add480 (480), dup79 (320),
dupB8 (160), dup480 (480). Duplicate controls copy existing Official Train
images already in training: they assess class weighting / training-row count,
NOT a new writer or independent data. Same seeds/writers/HP and preprocess.
No external438, Official Test or Professor Test. No deployed model updates.

CLI 인자 (정적 추출): `--arm`, `--seed`, `--heldout`, `--out`

In [ ]:
%%archive_source 05_model_optimize/MLP/Combined/scripts/exp_cmb011_classwise_training.py
#!/usr/bin/env python3
"""EXP-CMB-011: matched class-subset real handwriting vs official-train duplication controls.

Seven arms: base, add79 (320), addB8 (160), add480 (480), dup79 (320),
dupB8 (160), dup480 (480). Duplicate controls copy existing Official Train
images already in training: they assess class weighting / training-row count,
NOT a new writer or independent data. Same seeds/writers/HP and preprocess.
No external438, Official Test or Professor Test. No deployed model updates.
"""
from __future__ import annotations
import argparse,csv,hashlib,json,time,warnings
from pathlib import Path
import joblib,numpy as np,sklearn
from sklearn.pipeline import Pipeline
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import f1_score,confusion_matrix
from exp_cmb008_s5_early_stopping import (
    ROOT,WRITERS,LABELS,V3_SHA,HANDMADE_SHA,
    load_official,load_writers,PixelScaler255,prepareraw,process
)
EXTRA=ROOT/"02_data"/"Hand_made_extra"/"CMB010_7_8_9_B_480.npz"
EXPECTED_SHA="3d1686c67d7bb5c6193384eca1aa9218213c842252baa5b7e172698743b83fc7"
TARGET_COUNTS={7:160,8:80,9:160,11:80}
ARM_LABEL_COUNTS={
    "base":{},
    "add79":{7:160,9:160},
    "addB8":{8:80,11:80},
    "add480":TARGET_COUNTS,
    "dup79":{7:160,9:160},
    "dupB8":{8:80,11:80},
    "dup480":TARGET_COUNTS,
}
CONTROL_SELECTION_SEED=20261119
OFFICIAL=ROOT/"02_data"/"datasets"/"MNIST_EMNIST_official_split_112000.npz"
HANDMADE=ROOT/"02_data"/"Hand_made"
PAIRS=((7,9,"7_to_9"),(9,7,"9_to_7"),(11,8,"B_to_8"),
       (8,11,"8_to_B"),(0,13,"0_to_D"),(13,0,"D_to_0"),
       (7,10,"7_to_A"),(4,15,"4_to_F"))

def sha_file(path):
    h=hashlib.sha256()
    with path.open("rb") as f:
        for b in iter(lambda:f.read(1024*1024),b""):h.update(b)
    return h.hexdigest()

def validate_extra():
    assert sha_file(EXTRA)==EXPECTED_SHA,"New input NPZ unexpectedly changed"
    with np.load(EXTRA,allow_pickle=False) as z:
        required={"X","y","images","labels","label_text","class_names","available_classes","source_image","cell_row_col"}
        assert required.issubset(z.files),(required-set(z.files))
        x=np.asarray(z["X"]);y=np.asarray(z["y"],dtype=np.int32)
        assert x.shape==(480,784) and x.dtype==np.uint8 and y.shape==(480,)
        assert np.array_equal(x,z["images"].reshape(480,784))
        assert np.array_equal(y,z["labels"])
        assert np.array_equal(z["label_text"],np.asarray([LABELS[k] for k in y]))
        assert np.array_equal(z["class_names"],np.asarray(list(LABELS)))
        assert {int(c):int(np.sum(y==c)) for c in sorted(set(y))}==TARGET_COUNTS
        assert len({row.tobytes() for row in x})==480,"Duplicate raw image"
        img=x.reshape(-1,28,28)
        mask=img>=32
        assert np.all(mask.reshape(480,-1).sum(axis=1)>=12),"Missing foreground"
        assert not(np.any(mask[:,0,:]) or np.any(mask[:,-1,:]) or
                   np.any(mask[:,:,0]) or np.any(mask[:,:,-1])),"Foreground clipped at edge"
        assert int(img.max())==206
        assert "writer_id" not in z.files and "writer_ids" not in z.files
    return x,y

def dump_csv(path,records):
    with path.open("w",newline="",encoding="utf-8-sig") as f:
        w=csv.DictWriter(f,fieldnames=list(records[0]))
        w.writeheader();w.writerows(records)

def metrics(y,p):
    cm=confusion_matrix(y,p,labels=np.arange(16))
    assert cm.sum()==len(y)
    return {"n":int(len(y)),"correct":int(np.sum(y==p)),
        "accuracy":float(np.mean(y==p)),
        "macro_f1":float(f1_score(y,p,average="macro",labels=np.arange(16),zero_division=0)),
        "class_support":{LABELS[c]:int(cm[c].sum()) for c in range(16)},
        "class_errors":{LABELS[c]:int(cm[c].sum()-cm[c,c]) for c in range(16)},
        "pairs":{name:int(cm[a,b]) for a,b,name in PAIRS},
        "digit_correct":int(np.sum(p[y<10]==y[y<10])),
        "letter_correct":int(np.sum(p[y>=10]==y[y>=10]))}

def main():
    ap=argparse.ArgumentParser()
    ap.add_argument("--arm",choices=tuple(ARM_LABEL_COUNTS),required=True)
    ap.add_argument("--seed",choices=(42,43,44),type=int,required=True)
    ap.add_argument("--heldout",choices=WRITERS,type=int,required=True)
    ap.add_argument("--out",type=Path,required=True)
    a=ap.parse_args()
    if a.out.exists():raise FileExistsError(a.out)
    a.out.mkdir(parents=True)
    begin=time.perf_counter()
    off=load_official(OFFICIAL);writers=load_writers(HANDMADE)
    assert off["sha256"]==V3_SHA
    for w in WRITERS:assert writers[w]["sha256"]==HANDMADE_SHA[w]
    new_raw,new_y=validate_extra()
    main_x,wr_x,base_aug,_=prepareraw(off,writers,"scale30_stroke28")
    remain=[w for w in WRITERS if w!=a.heldout]
    fit_x=[main_x]+[wr_x[w] for w in remain]
    fit_y=[np.asarray(off["y_train"],dtype=np.int32)]+[np.asarray(writers[w]["y"],dtype=np.int32) for w in remain]
    count_map=ARM_LABEL_COUNTS[a.arm]
    n_added=sum(count_map.values())
    is_new=a.arm.startswith("add")
    is_dup=a.arm.startswith("dup")
    duplicate_official_indices=[]
    if is_new:
        # Same original images and labels; select whole classes from the supplied NPZ.
        # Do NOT re-label, flip, or discard ambiguous examples post hoc.
        select=np.isin(new_y,list(count_map))
        assert int(select.sum())==n_added
        assert {int(c):int((new_y[select]==c).sum()) for c in count_map}==count_map
        new_x=process(new_raw[select],"com",20)
        fit_x.append(new_x)
        fit_y.append(new_y[select])
    elif is_dup:
        # Same-size and same-class-count control. All selected official examples
        # were already in the 76,800-row Official Training set; this ONLY tests
        # training-sample weighting, not addition of independent unique data.
        rng=np.random.default_rng(CONTROL_SELECTION_SEED)
        for cl,n in sorted(count_map.items()):
            pool=np.flatnonzero(np.asarray(off["y_train"])==cl)
            duplicate_official_indices.extend(rng.choice(pool,size=n,replace=False).tolist())
        ids=np.asarray(duplicate_official_indices,dtype=int)
        assert len(ids)==n_added and len(set(ids))==n_added
        fit_x.append(main_x[ids])
        fit_y.append(np.asarray(off["y_train"],dtype=int)[ids])
    fit_x=np.concatenate(fit_x)
    fit_y=np.concatenate(fit_y)
    expected=78720+n_added
    assert fit_x.shape==(expected,784) and fit_y.shape==(expected,)
    held_x=wr_x[a.heldout]
    held_y=np.asarray(writers[a.heldout]["y"],dtype=int)
    val_x=process(off["X_val"],"com",20)
    val_y=np.asarray(off["y_val"],dtype=int)
    assert len(held_y)==640 and len(val_y)==19200
    mlp=MLPClassifier(random_state=a.seed,hidden_layer_sizes=(256,),
        alpha=.0001,learning_rate_init=.001,solver="adam",activation="relu",
        early_stopping=False,batch_size=128,max_iter=200,
        n_iter_no_change=10,tol=.0001)
    model=Pipeline([("scale_0_1",PixelScaler255()),("model",mlp)])
    with warnings.catch_warnings(record=True) as got:
        warnings.simplefilter("always")
        tick=time.perf_counter()
        model.fit(fit_x,fit_y)
        fit_sec=time.perf_counter()-tick
    assert np.array_equal(mlp.classes_,np.arange(16))
    op=np.asarray(model.predict(held_x),dtype=int)
    vp=np.asarray(model.predict(val_x),dtype=int)
    oof=metrics(held_y,op)
    val=metrics(val_y,vp)
    dump_csv(a.out/"heldout_predictions.csv",
        [{"arm":a.arm,"seed":a.seed,"heldout":a.heldout,
          "index":i,"actual":LABELS[int(held_y[i])],
          "pred":LABELS[int(op[i])]} for i in range(640)])
    # Optional fitted models preserved only in GitHub Actions artifacts, not deployed.
    joblib.dump(model,a.out/"candidate_model.joblib",compress=3)
    m={"experiment":"EXP-CMB-011","arm":a.arm,"seed":a.seed,"heldout":a.heldout,
       "official_sha256":off["sha256"],
       "old_writers_sha256":{str(w):writers[w]["sha256"] for w in WRITERS},
       "added_source_sha256":EXPECTED_SHA,
       "new_cohort_writer_id_known":False,"new_cohort_grouped_as_one":True,
       "added_images_count":n_added if is_new else 0,
       "duplicated_training_rows_count":n_added if is_dup else 0,
       "added_by_class":{LABELS[k]:v if is_new else 0 for k,v in count_map.items()},
       "duplicated_by_class":{LABELS[k]:v if is_dup else 0 for k,v in count_map.items()},
       "duplicate_selection_seed":CONTROL_SELECTION_SEED if is_dup else None,
       "duplicate_official_indices_sha256":hashlib.sha256(np.asarray(duplicate_official_indices,dtype=np.int64).tobytes()).hexdigest() if is_dup else None,
       "control_explanation":"Repeated existing official-train images; no independent new examples" if is_dup else None,
       "training_samples":expected,"heldout_samples":len(held_y),
       "official_val_samples":len(val_y),"heldout_in_fit":False,
       "new_cohort_in_fit":is_new,
       "new_cohort_tested_as_independent":False,
       "external_438_used":False,"original_test_used":False,"professor_test_used":False,
       "frozen":{"hidden_layer_sizes":[256],"alpha":.0001,"learning_rate_init":.001,
                 "activation":"relu","solver":"adam","early_stopping":False,
                 "batch_size":128,"max_iter":200,"n_iter_no_change":10,"tol":.0001,
                 "official_aug":"scale30_stroke28","inference_preprocess":"CoM"},
       "augmentation":base_aug,"oof":oof,"official_val":val,
       "n_iter":int(mlp.n_iter_),"fit_seconds":float(fit_sec),
       "total_seconds":float(time.perf_counter()-begin),
       "sklearn_version":sklearn.__version__,
       "warnings":[str(w.message) for w in got],
       "model_sha256":sha_file(a.out/"candidate_model.joblib")}
    (a.out/"manifest.json").write_text(json.dumps(m,ensure_ascii=False,indent=2)+"\n",encoding="utf-8")
    print("CMB011_FINISHED",json.dumps({"arm":a.arm,"seed":a.seed,
        "heldout":a.heldout,"fit_count":expected,"oof":oof["accuracy"],
        "val":val["accuracy"],"7_to_9":oof["pairs"]["7_to_9"],
        "B_to_8":oof["pairs"]["B_to_8"],"fit_seconds":round(fit_sec,1)}),flush=True)

if __name__=="__main__":main()


## exp_cmb010_aggregate

원본: [05_model_optimize/MLP/Combined/scripts/exp_cmb010_aggregate.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/scripts/exp_cmb010_aggregate.py)  
Git blob: `f0d3a22452059adbcd63c6ec01eb801ddf88f63b`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/logs)

EXP-CMB-010 paired 12-run baseline/add480 aggregate; no test-based selection.

CLI 인자 (정적 추출): `--incoming`, `--out`

In [ ]:
%%archive_source 05_model_optimize/MLP/Combined/scripts/exp_cmb010_aggregate.py
#!/usr/bin/env python3
"""EXP-CMB-010 paired 12-run baseline/add480 aggregate; no test-based selection."""
import argparse,csv,hashlib,json
from collections import defaultdict
from pathlib import Path
ARMS=("base","add480")
SEEDS=(42,43,44)
WRITERS=(41,42,43,44)
LABELS="0123456789ABCDEF"
WATCH=("7_to_9","9_to_7","B_to_8","8_to_B","0_to_D","D_to_0","7_to_A","4_to_F")
def dump(path,o):
    path.write_text(json.dumps(o,ensure_ascii=False,indent=2)+"\n",encoding="utf-8")
def load(incoming):
    rec,lines={},{}
    for arm in ARMS:
        for seed in SEEDS:
            for writer in WRITERS:
                key=(arm,seed,writer)
                folder=incoming/f"cmb010-{arm}-s{seed}-w{writer}"
                m=json.loads((folder/"manifest.json").read_text())
                assert (m["arm"],m["seed"],m["heldout"])==key
                assert m["training_samples"]==(78720+(480 if arm=="add480" else 0))
                assert m["heldout_samples"]==640 and m["official_val_samples"]==19200
                assert m["frozen"]["batch_size"]==128
                assert not m["heldout_in_fit"] and not m["external_438_used"]
                assert not m["original_test_used"] and not m["professor_test_used"]
                assert m["added_source_sha256"]=="3d1686c67d7bb5c6193384eca1aa9218213c842252baa5b7e172698743b83fc7"
                assert hashlib.sha256((folder/"candidate_model.joblib").read_bytes()).hexdigest()==m["model_sha256"]
                with (folder/"heldout_predictions.csv").open(newline="",encoding="utf-8-sig") as f:
                    rows=list(csv.DictReader(f))
                assert len(rows)==640 and [int(r["index"]) for r in rows]==list(range(640))
                rec[key]=m;lines[key]=rows
    for seed in SEEDS:
        for w in WRITERS:
            m0,m1=rec[("base",seed,w)],rec[("add480",seed,w)]
            assert m0["official_sha256"]==m1["official_sha256"]
            assert m0["old_writers_sha256"]==m1["old_writers_sha256"]
            assert [r["actual"] for r in lines[("base",seed,w)]]==[r["actual"] for r in lines[("add480",seed,w)]]
    return rec,lines
def summarize(rec):
    result={}
    for arm in ARMS:
        rows=[v for k,v in rec.items() if k[0]==arm]
        def avg(data,key):
            return sum(x[data][key] for x in rows)/len(rows)
        result[arm]={"n_models":len(rows),"training_samples_per_model":rows[0]["training_samples"],
            "oof_accuracy":avg("oof","accuracy"),
            "oof_macro_f1":avg("oof","macro_f1"),
            "official_val_accuracy":avg("official_val","accuracy"),
            "official_val_macro_f1":avg("official_val","macro_f1"),
            "writer_accuracy":{str(w):sum(rec[(arm,s,w)]["oof"]["accuracy"] for s in SEEDS)/3 for w in WRITERS},
            "seed_accuracy":{str(s):sum(rec[(arm,s,w)]["oof"]["accuracy"] for w in WRITERS)/4 for s in SEEDS},
            "pairs":{p:sum(x["oof"]["pairs"][p] for x in rows) for p in WATCH},
            "class_errors":{c:sum(x["oof"]["class_errors"][c] for x in rows) for c in LABELS},
            "class_support":{c:sum(x["oof"]["class_support"][c] for x in rows) for c in LABELS},
            "total_oof_correct":sum(x["oof"]["correct"] for x in rows),
            "oof_evaluation_trials":len(rows)*640,
            "n_unique_oof_images":4*640,
            "n_unique_official_val_images":19200}
    return result

def paired(rows):
    corr=reg=0
    class_corr=defaultdict(int);class_reg=defaultdict(int);direction=defaultdict(int)
    for seed in SEEDS:
        for writer in WRITERS:
            x0=rows[("base",seed,writer)]
            x1=rows[("add480",seed,writer)]
            for a,b in zip(x0,x1):
                assert (a["index"],a["actual"])==(b["index"],b["actual"])
                label=a["actual"]
                old=a["pred"]==label; new=b["pred"]==label
                if not old and new:corr+=1;class_corr[label]+=1
                if old and not new:reg+=1;class_reg[label]+=1
                if a["pred"]!=b["pred"]:direction[f"{a['pred']}->{b['pred']}"]+=1
    return {"corrected":corr,"new_errors":reg,"net_correct":corr-reg,
            "corrected_by_actual_class":dict(class_corr),
            "new_errors_by_actual_class":dict(class_reg)}

def decision(s):
    a,b=s["base"],s["add480"]
    tests={"oof_accuracy_non_decrease":b["oof_accuracy"]>=a["oof_accuracy"],
           "official_val_drop_not_more_than_0p3pp":(a["official_val_accuracy"]-b["official_val_accuracy"])<=.003,
           "7_to_9_decrease":b["pairs"]["7_to_9"]<a["pairs"]["7_to_9"],
           "9_to_7_nonincrease":b["pairs"]["9_to_7"]<=a["pairs"]["9_to_7"],
           "B_to_8_nonincrease":b["pairs"]["B_to_8"]<=a["pairs"]["B_to_8"],
           "8_to_B_nonincrease":b["pairs"]["8_to_B"]<=a["pairs"]["8_to_B"],
           "D_to_0_nonincrease":b["pairs"]["D_to_0"]<=a["pairs"]["D_to_0"],
           "0_to_D_nonincrease":b["pairs"]["0_to_D"]<=a["pairs"]["0_to_D"]}
    return {"exploratory_predeclared_checks":tests,"pass_all":all(tests.values()),
            "oof_gain_pp":100*(b["oof_accuracy"]-a["oof_accuracy"]),
            "official_val_gain_pp":100*(b["official_val_accuracy"]-a["official_val_accuracy"]),
            "limitations":["Single unknown-writer cohort; cannot prove distinct from writers41-44",
                           "Existing OOF writers reused repeatedly; development-only, not independent",
                           "Class-targeted data additions can alter prior probabilities",
                           "No untouched independent 16-class writer-test evaluation"],
            "deployment_decision":"No automatic deployment or main merge"}
def main():
    a=argparse.ArgumentParser()
    a.add_argument("--incoming",type=Path,required=True)
    a.add_argument("--out",type=Path,required=True)
    z=a.parse_args()
    z.out.mkdir(parents=True,exist_ok=True)
    rec,pred=load(z.incoming)
    s=summarize(rec);delta=paired(pred);gate=decision(s)
    report={"experiment":"EXP-CMB-010","source_images_by_class":{"7":160,"8":80,"9":160,"B":80},
      "source_sha256":"3d1686c67d7bb5c6193384eca1aa9218213c842252baa5b7e172698743b83fc7",
      "new_cohort_writer_id_known":False,
      "n_fits":24,"arms":s,"paired":delta,"preliminary_gates":gate}
    dump(z.out/"summary.json",report)
    fields=["arm","oof_accuracy","official_val_accuracy","oof_macro_f1","official_val_macro_f1",
            *WATCH,"n_models","training_samples_per_model"]
    with (z.out/"summary.csv").open("w",newline="",encoding="utf-8-sig") as f:
        w=csv.DictWriter(f,fieldnames=fields);w.writeheader()
        for arm in ARMS:
            row=s[arm]
            w.writerow({"arm":arm,**{k:row[k] for k in fields if k in row},
                        **row["pairs"]})
    with (z.out/"class_error_comparison.csv").open("w",newline="",encoding="utf-8-sig") as f:
        w=csv.DictWriter(f,fieldnames=["class","support_total","base_errors","add480_errors",
            "delta_errors","base_error_rate","add480_error_rate"]);w.writeheader()
        for c in LABELS:
            denom=s["base"]["class_support"][c]
            assert denom==s["add480"]["class_support"][c]
            x=s["base"]["class_errors"][c];y=s["add480"]["class_errors"][c]
            w.writerow({"class":c,"support_total":denom,"base_errors":x,"add480_errors":y,
                "delta_errors":y-x,"base_error_rate":x/denom,"add480_error_rate":y/denom})
    print("CMB010_AGGREGATE",json.dumps({"base_oof":s["base"]["oof_accuracy"],
      "new_oof":s["add480"]["oof_accuracy"],
      "base_val":s["base"]["official_val_accuracy"],
      "new_val":s["add480"]["official_val_accuracy"],
      "paired":delta,"gates":gate["exploratory_predeclared_checks"]},ensure_ascii=False),flush=True)
if __name__=="__main__":main()


## exp_cmb011_aggregate

원본: [05_model_optimize/MLP/Combined/scripts/exp_cmb011_aggregate.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/scripts/exp_cmb011_aggregate.py)  
Git blob: `e897d9821753d6579701efb588ce7560571f998d`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/logs)

CMB011: 7-arm writer-heldout paired audit, class-balanced controls and P/R/F1.

CLI 인자 (정적 추출): `--incoming`, `--out`

In [ ]:
%%archive_source 05_model_optimize/MLP/Combined/scripts/exp_cmb011_aggregate.py
#!/usr/bin/env python3
"""CMB011: 7-arm writer-heldout paired audit, class-balanced controls and P/R/F1."""
from __future__ import annotations
import argparse,csv,hashlib,json
from collections import defaultdict
from pathlib import Path
import numpy as np
from sklearn.metrics import precision_recall_fscore_support,confusion_matrix
ARMS=("base","add79","addB8","add480","dup79","dupB8","dup480")
COUNT={"base":0,"add79":320,"addB8":160,"add480":480,"dup79":320,"dupB8":160,"dup480":480}
SEEDS=(42,43,44)
WRITERS=(41,42,43,44)
LABELS="0123456789ABCDEF"
PAIRS=((7,9,"7_to_9"),(9,7,"9_to_7"),(11,8,"B_to_8"),(8,11,"8_to_B"),(0,13,"0_to_D"),(13,0,"D_to_0"),(7,10,"7_to_A"),(4,15,"4_to_F"))
SOURCE_SHA="3d1686c67d7bb5c6193384eca1aa9218213c842252baa5b7e172698743b83fc7"
def writecsv(p,rows):
    assert rows
    with p.open("w",newline="",encoding="utf-8-sig") as f:
        w=csv.DictWriter(f,fieldnames=list(rows[0]));w.writeheader();w.writerows(rows)
def readall(source):
    m,pp={},{}
    for arm in ARMS:
        for seed in SEEDS:
            for writer in WRITERS:
                k=(arm,seed,writer)
                folder=source/f"cmb011-{arm}-s{seed}-w{writer}"
                man=json.loads((folder/"manifest.json").read_text(encoding="utf-8"))
                assert (man["arm"],int(man["seed"]),int(man["heldout"]))==k
                assert man["experiment"]=="EXP-CMB-011"
                assert man["training_samples"]==78720+COUNT[arm]
                assert man["added_source_sha256"]==SOURCE_SHA
                assert man["heldout_samples"]==640 and man["official_val_samples"]==19200
                assert man["frozen"]["batch_size"]==128
                assert not man["heldout_in_fit"] and not man["external_438_used"]
                assert not man["original_test_used"] and not man["professor_test_used"]
                assert man["new_cohort_in_fit"]==arm.startswith("add")
                assert man["added_images_count"]==(COUNT[arm] if arm.startswith("add") else 0)
                assert man["duplicated_training_rows_count"]==(COUNT[arm] if arm.startswith("dup") else 0)
                with (folder/"heldout_predictions.csv").open(newline="",encoding="utf-8-sig") as f:
                    rows=list(csv.DictReader(f))
                assert len(rows)==640 and [int(v["index"]) for v in rows]==list(range(640))
                assert all((v["arm"],int(v["seed"]),int(v["heldout"]))==k for v in rows)
                actual=np.asarray([v["actual"] for v in rows])
                pred=np.asarray([v["pred"] for v in rows])
                assert np.isclose((actual==pred).mean(),man["oof"]["accuracy"])
                m[k]=man;pp[k]=(actual,pred)
    for seed in SEEDS:
        for writer in WRITERS:
            original=pp[("base",seed,writer)][0]
            for arm in ARMS[1:]:
                assert np.array_equal(original,pp[(arm,seed,writer)][0])
                assert m[("base",seed,writer)]["official_sha256"]==m[(arm,seed,writer)]["official_sha256"]
                assert m[("base",seed,writer)]["old_writers_sha256"]==m[(arm,seed,writer)]["old_writers_sha256"]
    return m,pp
def compute(m,pp):
    allsum={};classrows=[];foldrows=[]
    for arm in ARMS:
        mats=[]; foldmetrics=[]
        for seed in SEEDS:
            for writer in WRITERS:
                actual,pred=pp[(arm,seed,writer)]
                pr,re,f1,_=precision_recall_fscore_support(actual,pred,labels=list(LABELS),zero_division=0)
                foldmetrics.append([float(pr.mean()),float(re.mean()),float(f1.mean())])
                foldrows.append({"arm":arm,"seed":seed,"heldout_writer":writer,
                                 "accuracy":float((actual==pred).mean()),
                                 "macro_precision":float(pr.mean()),
                                 "macro_recall":float(re.mean()),"macro_f1":float(f1.mean()),
                                 "official_val_accuracy":m[(arm,seed,writer)]["official_val"]["accuracy"],
                                 "official_val_macro_f1":m[(arm,seed,writer)]["official_val"]["macro_f1"],
                                 "fit_seconds":m[(arm,seed,writer)]["fit_seconds"],
                                 "n_iter":m[(arm,seed,writer)]["n_iter"]})
                mats.append(confusion_matrix(actual,pred,labels=list(LABELS)))
        cm=np.sum(mats,axis=0)
        assert cm.sum()==7680
        pre,rec,f,_=precision_recall_fscore_support(
            np.repeat(np.arange(16),cm.sum(axis=1)),
            np.concatenate([np.repeat(j,cm[i,j]) for i in range(16) for j in range(16)]),
            labels=np.arange(16),zero_division=0) if False else (None,None,None,None)
        p_cls=np.diag(cm)/np.maximum(cm.sum(axis=0),1)
        r_cls=np.diag(cm)/np.maximum(cm.sum(axis=1),1)
        f_cls=np.divide(2*p_cls*r_cls,p_cls+r_cls,out=np.zeros_like(p_cls,dtype=float),where=(p_cls+r_cls)>0)
        for i,cl in enumerate(LABELS):
            classrows.append({"arm":arm,"class":cl,"support":int(cm[i].sum()),
                "precision":float(p_cls[i]),"recall":float(r_cls[i]),"f1":float(f_cls[i]),
                "errors":int(cm[i].sum()-cm[i,i])})
        foldarr=np.asarray(foldmetrics)
        av=lambda key:sum(m[(arm,s,w)][key]["accuracy"] for s in SEEDS for w in WRITERS)/12
        avgvalf1=sum(m[(arm,s,w)]["official_val"]["macro_f1"] for s in SEEDS for w in WRITERS)/12
        total_correct=int(np.trace(cm))
        allsum[arm]={"n_models":12,"train_rows":78720+COUNT[arm],
          "new_handmade_rows":COUNT[arm] if arm.startswith("add") else 0,
          "duplicated_official_rows":COUNT[arm] if arm.startswith("dup") else 0,
          "oof_accuracy":float(total_correct/7680),
          "macro_precision":float(foldarr[:,0].mean()),
          "macro_recall":float(foldarr[:,1].mean()),
          "macro_f1":float(foldarr[:,2].mean()),
          "official_val_accuracy":float(av("official_val")),
          "official_val_macro_f1":float(avgvalf1),
          "correct":total_correct,
          "class_errors":{LABELS[i]:int(cm[i].sum()-cm[i,i]) for i in range(16)},
          "class_f1":{LABELS[i]:float(f_cls[i]) for i in range(16)},
          "pairs":{name:int(cm[a,b]) for a,b,name in PAIRS},
          "writer_accuracy":{str(w):float(np.mean([m[(arm,s,w)]["oof"]["accuracy"] for s in SEEDS])) for w in WRITERS},
          "seed_accuracy":{str(s):float(np.mean([m[(arm,s,w)]["oof"]["accuracy"] for w in WRITERS])) for s in SEEDS},
          "confusion_16class":cm.tolist()}
    paired={}
    for arm in ARMS[1:]:
        corrected=regress=0
        byclass={c:{"corrected":0,"new_errors":0} for c in LABELS}
        for seed in SEEDS:
            for w in WRITERS:
                actual,b=pp[("base",seed,w)]
                a2,p=pp[(arm,seed,w)]
                assert np.array_equal(actual,a2)
                fix=(b!=actual)&(p==actual)
                broken=(b==actual)&(p!=actual)
                corrected+=int(fix.sum());regress+=int(broken.sum())
                for c in LABELS:
                    t=actual==c
                    byclass[c]["corrected"]+=int((t&fix).sum())
                    byclass[c]["new_errors"]+=int((t&broken).sum())
        paired[arm]={"corrected":corrected,"new_errors":regress,
                     "net_correct":corrected-regress,"per_class":byclass,
                     "oof_delta_pp":100*(allsum[arm]["oof_accuracy"]-allsum["base"]["oof_accuracy"]),
                     "official_val_delta_pp":100*(allsum[arm]["official_val_accuracy"]-allsum["base"]["official_val_accuracy"])}
    return allsum,paired,foldrows,classrows
def main():
    ap=argparse.ArgumentParser()
    ap.add_argument("--incoming",required=True,type=Path)
    ap.add_argument("--out",required=True,type=Path)
    a=ap.parse_args()
    a.out.mkdir(parents=True,exist_ok=True)
    m,pp=readall(a.incoming)
    summary,paired,folds,classes=compute(m,pp)
    result={"experiment":"EXP-CMB-011",
      "study":"class partition and same-count official-train duplication controls",
      "caveats":["Existing Official Train duplication is sample reweighting, not extra unique data",
                 "Original new writer IDs unknown; 480 grouped as one cohort",
                 "Repeated OOF writer set used for selection in past; not independent test",
                 "No Original Test, Professor Test, external 438 used for training or selection"],
      "n_models":len(m),"n_unique_oof_images":2560,"oof_trials_per_arm":7680,
      "arms":summary,"paired_vs_baseline":paired}
    (a.out/"summary.json").write_text(json.dumps(result,ensure_ascii=False,indent=2)+"\n",encoding="utf-8")
    names=("arm","train_rows","new_handmade_rows","duplicated_official_rows","oof_accuracy","macro_precision","macro_recall","macro_f1","official_val_accuracy","official_val_macro_f1","correct")
    writecsv(a.out/"summary.csv",[{"arm":arm,**{q:summary[arm][q] for q in names if q!="arm"}} for arm in ARMS])
    writecsv(a.out/"per_class.csv",classes)
    writecsv(a.out/"per_fold.csv",folds)
    writecsv(a.out/"paired.csv",[{"arm":arm,**{k:v for k,v in paired[arm].items() if k!="per_class"}} for arm in ARMS[1:]])
    rows=[]
    for arm in ARMS:
        for k,v in summary[arm]["pairs"].items():
            rows.append({"arm":arm,"direction":k,"errors":v})
    writecsv(a.out/"focus_pairs.csv",rows)
    print("CMB011_FINISHED",json.dumps({"n_models":len(m),"arms":{k:{"oof":v["oof_accuracy"],"macro_f1":v["macro_f1"],"val":v["official_val_accuracy"],"pairs":v["pairs"]} for k,v in summary.items()},"paired":{k:{"net_correct":v["net_correct"]} for k,v in paired.items()}},ensure_ascii=False),flush=True)
if __name__=="__main__":
    main()


## 실험 실행 및 결과 확인

`RUN_PLAN`에 실제 코드의 CLI 인자와 실행 순서를 넣고, `ALLOW_EXECUTION=True`를 설정한 경우에만 실행된다. 이전 결과를 다시 계산한 것과 새 학습 결과를 구별한다.

In [ ]:
if RUN_PLAN:
    for source_path, arguments in RUN_PLAN:
        execute_original(source_path, arguments)
else:
    print("원본 코드 등록 완료. 학습/평가는 실행하지 않았습니다.")